In [1]:
%load_ext autoreload
%autoreload 2

import os
import torch

torch.set_float32_matmul_precision("medium")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TORCHDYNAMO_DISABLE"] = "1"  # avoid triton/inductor version mismatch

## Configuration
Set the wandb run ID and MTEB tasks to evaluate on. The task type (ae/declutr) is auto-detected from the wandb config.

In [7]:
# === Configure these ===
WANDB_ID = "5rg719g4"  # wandb run ID
DEVICE = "cuda"
BATCH_SIZE = 64

# MTEB tasks to run (a representative subset)
MTEB_TASKS = [
    # STS
    "STSBenchmark",
    #"SICK-R",
    # Classification
    "Banking77Classification.v2",
    #"ToxicConversationsClassification",
    # Clustering
    #"ArXivHierarchicalClusteringS2S",
    #"RedditClustering",
    # Retrieval
    #"SciFact",
    #"NFCorpus",
]

## Load encoder from checkpoint

In [8]:
from mteb_wrapper import MTEBEncoderWrapper, load_encoder_from_checkpoint

encoder = load_encoder_from_checkpoint(
    wandb_id=WANDB_ID,
    device=DEVICE,
)

print(f"Encoder loaded: backbone={encoder.cfg.model_name}")
if encoder.cfg.sem is not None:
    print(f"SEM config: L={encoder.sem.cfg.L}, V={encoder.sem.cfg.V}")
    print(f"Embedding dim (L*V): {encoder.sem.cfg.L * encoder.sem.cfg.V}")

Detected task type: declutr
Encoder loaded: backbone=FacebookAI/roberta-large
SEM config: L=512, V=16
Embedding dim (L*V): 8192


In [47]:
from model.encoder import EncoderModel, EncoderConfig

encoder = EncoderModel(EncoderConfig(
    model_name="FacebookAI/roberta-large",
    no_out_proj=True,
))

## Run MTEB evaluation
Evaluates in both **soft** (standard temperature) and **hard** (near-zero temperature, one-hot) modes.

In [9]:
import mteb
from copy import deepcopy

tasks = mteb.get_tasks(tasks=MTEB_TASKS)
results = {}
modes = ["soft", "hard"] if encoder.cfg.sem is not None else ["soft"]  # only evaluate "soft" if SEM is present
for mode in modes:
    print(f"\n{'='*60}")
    print(f"Evaluating mode: {mode}")
    print(f"{'='*60}\n")

    model = MTEBEncoderWrapper(
        encoder=encoder,
        mode=mode,
        batch_size=BATCH_SIZE,
        device=DEVICE
    )

    results[mode] = mteb.evaluate(model=model, tasks=deepcopy(tasks))
print("\nDone!")


Evaluating mode: soft



Evaluating tasks:   0%|          | 0/2 [00:00<?, ?it/s]


Evaluating mode: hard



Evaluating tasks:   0%|          | 0/2 [00:00<?, ?it/s]


Done!


## Compare soft vs hard results

In [10]:
import pandas as pd


def extract_main_score(task_results):
    """Extract the main score from MTEB task results."""
    scores = {}
    for task_result in task_results:
        task_name = task_result.task_name
        for split in ["test", "dev", "validation"]:
            if split in task_result.scores:
                score_list = task_result.scores[split]
                if score_list:
                    scores[task_name] = score_list[0].get("main_score", None)
                    break
    return scores


soft_scores = extract_main_score(results["soft"])
if "hard" in results:
    hard_scores = extract_main_score(results["hard"])
    df_dict = {
        "Task":  list(soft_scores.keys()),
        "Soft":  [soft_scores[t]     for t in soft_scores],
        "Hard":  [hard_scores.get(t) for t in soft_scores],
    }
else:
    df_dict = {
        "Task":  list(soft_scores.keys()),
        "Soft":  [soft_scores[t]     for t in soft_scores],
    }

df = pd.DataFrame(
    df_dict
).set_index("Task")

if "Hard" in df.columns:
    df["Delta"] = df["Soft"] - df["Hard"]

print(df.to_string(float_format="{:.4f}".format))
print(f"\nAverage soft:  {df['Soft'].mean():.4f}")
if "Hard" in df.columns:
    print(f"Average hard:  {df['Hard'].mean():.4f}")
    print(f"Average delta: {df['Delta'].mean():.4f}")

                             Soft   Hard  Delta
Task                                           
STSBenchmark               0.7370 0.7168 0.0202
Banking77Classification.v2 0.7430 0.7169 0.0261

Average soft:  0.7400
Average hard:  0.7169
Average delta: 0.0231
